# 🔎 02 — Vani: Build Databricks Vector Search Index

Creates a **Databricks Vector Search** endpoint + Delta-synced index over `silver.rules_chunks`.

Why this is a huge win for scoring:
- Vector Search is **THE showcase GenAI feature on Databricks** — using it scores well under Databricks Usage (30%)
- Delta-sync means the index updates automatically as we add more rules — full Lakehouse integration
- Judges can query the index directly from SQL in Genie
- No manual embedding calls from Python → production-grade architecture

In [0]:
%pip install -q databricks-vectorsearch
%restart_python

In [0]:
from databricks.vector_search.client import VectorSearchClient

ENDPOINT_NAME   = "setu_rail_vs"
INDEX_NAME      = "setu_rail.gold.rules_vs_index"
SOURCE_TABLE    = "setu_rail.silver.rules_chunks"

client = VectorSearchClient()

In [0]:
# --- Create the endpoint (skip if exists) ---
try:
    client.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    print(f"✅ Endpoint '{ENDPOINT_NAME}' creation requested (may take ~5 min to be ready)")
except Exception as e:
    print(f"Endpoint already exists or error: {e}")

# Wait for the endpoint to come online
client.wait_for_endpoint(ENDPOINT_NAME, timeout=600)
print("✅ Endpoint is ONLINE")

In [0]:
# --- Create Delta-sync index (auto-embed with databricks-bge-large-en) ---
try:
    index = client.create_delta_sync_index(
        endpoint_name           = ENDPOINT_NAME,
        index_name              = INDEX_NAME,
        source_table_name       = SOURCE_TABLE,
        primary_key             = "id",
        embedding_source_column = "text",
        embedding_model_endpoint_name = "databricks-bge-large-en",
        pipeline_type           = "TRIGGERED",  # manual sync; switch to CONTINUOUS in prod
    )
    print(f"✅ Index '{INDEX_NAME}' creation started — sync running")
except Exception as e:
    print(f"Index already exists or error: {e}")

In [0]:
# --- Wait for index sync to finish ---
import time
idx = client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
for _ in range(60):   # up to 10 min
    desc = idx.describe()
    status = desc.get("status", {}).get("detailed_state", "UNKNOWN")
    print(f"Index status: {status}")
    if "ONLINE" in status:
        break
    time.sleep(10)
print("✅ Index is ONLINE")

In [0]:
# --- Test retrieval with a passenger-rights question ---
test_queries = [
    "What compensation am I entitled to if my train is delayed for hours?",
    "Can I get a refund if I cancel my ticket?",
    "What happens if there is an accident on the train?",
]

for q in test_queries:
    print(f"\n🔎 Query: {q}")
    results = idx.similarity_search(
        query_text    = q,
        columns       = ["id", "source", "source_title", "page", "section", "text"],
        num_results   = 3,
    )
    for row in results["result"]["data_array"]:
        _, source, title, page, section, text, _score = row
        print(f"  → {source} p.{page}  §{section}  (score {_score:.3f})")
        print(f"     {text[:120]}...")

✅ **Next:** `03_vani_rag_agent.ipynb`